# AgentCore 온라인 평가

프로그래밍 방식의 테스트를 위한 에이전트 호출 및 평가 워크플로입니다.

## 가져오기

In [ ]:
import boto3
from IPython.display import Markdown, display
from utils import (
    EvaluationClient,
    generate_session_id,
    invoke_and_evaluate,
)

# AWS 자격 증명을 설정합니다.

# os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
# os.environ['AWS_ACCESS_KEY_ID'] = ''
# os.environ['AWS_SECRET_ACCESS_KEY'] = ''
# os.environ['AWS_SESSION_TOKEN'] = ''

## 구성

In [ ]:
AGENT_ID = "strands_claude_eval"
AGENT_ARN = f"arn:aws:bedrock-agentcore:us-east-1:<YOUR_ACCOUNT_ID>:runtime/{AGENT_ID}"
REGION = "us-east-1"

## 실험 설정

In [ ]:
EXPERIMENT_NAME = "my_experiment_v1"

# 13개 evaluator를 모두 실행하려면 None으로 설정합니다(종합 모드).
# 일부만 실행하려면 FLEXIBLE_EVALUATORS 같은 특정 목록으로 설정합니다.
EXPERIMENT_EVALUATORS = None  # 13개 evaluator를 모두 실행합니다.

EXPERIMENT_SCOPE = "session"  # EXPERIMENT_EVALUATORS가 None이면 무시됩니다.
EXPERIMENT_DELAY = 120  # trace가 AgentCore Observability에 도달하도록 120초 이상 기다립니다.

planned_session = generate_session_id()

# metadata는 선택 사항이지만 추적에 유용합니다.
EXPERIMENT_PROMPTS = [
    {"prompt": "What is 2 + 2?", "session_id": "", "metadata": {"category": "math"}},
    {
        "prompt": "What is the capital of France?",
        "session_id": "",
        "metadata": {"category": "geography"},
    },
    {
        "prompt": "Tell me about quantum physics",
        "session_id": "",
        "metadata": {"category": "science"},
    },
    {
        "prompt": "Hello, can you help me with math?",
        "session_id": planned_session,
        "metadata": {"turn": 1},
    },
    {
        "prompt": "What is 15 * 23?",
        "session_id": planned_session,
        "metadata": {"turn": 2},
    },
]

## 클라이언트 초기화

In [ ]:
# AgentCore 클라이언트를 초기화합니다.
agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)

# 평가 클라이언트를 초기화합니다.
eval_client = EvaluationClient(
    region=REGION,
)

## 실험 실행

In [ ]:
FLEXIBLE_EVALUATORS = [
    "Builtin.Correctness",
    "Builtin.Faithfulness",
    "Builtin.Helpfulness",
    "Builtin.ResponseRelevance",
    "Builtin.Conciseness",
    "Builtin.Coherence",
    "Builtin.InstructionFollowing",
    "Builtin.Refusal",
    "Builtin.Harmfulness",
    "Builtin.Stereotyping",
]

SESSION_ONLY_EVALUATORS = ["Builtin.GoalSuccessRate"]

SPAN_ONLY_EVALUATORS = [
    "Builtin.ToolSelectionAccuracy",
    "Builtin.ToolParameterAccuracy",
]

In [ ]:
batch_results = []

eval_count = len(EXPERIMENT_EVALUATORS) if EXPERIMENT_EVALUATORS else 13
print(f"Experiment: {EXPERIMENT_NAME} | Prompts: {len(EXPERIMENT_PROMPTS)} | Evaluators: {eval_count}\n")

for i, config in enumerate(EXPERIMENT_PROMPTS, 1):
    prompt_text = config["prompt"]
    session_id = config.get("session_id", "")
    metadata = config.get("metadata", {})

    try:
        returned_session_id, content, results = invoke_and_evaluate(
            agentcore_client=agentcore_client,
            eval_client=eval_client,
            agent_arn=AGENT_ARN,
            agent_id=AGENT_ID,
            region=REGION,
            prompt=prompt_text,
            experiment_name=EXPERIMENT_NAME,
            session_id=session_id,
            metadata=metadata,
            evaluators=EXPERIMENT_EVALUATORS,
            scope=EXPERIMENT_SCOPE,
            delay=EXPERIMENT_DELAY,
            flexible_evaluators=FLEXIBLE_EVALUATORS,
            session_only_evaluators=SESSION_ONLY_EVALUATORS,
            span_only_evaluators=SPAN_ONLY_EVALUATORS,
        )

        if content:
            display(Markdown(str(content[0])))

        batch_results.append(
            {
                "session_id": returned_session_id,
                "prompt": prompt_text,
                "results": results,
            }
        )

    except Exception as e:
        print(f"Error: {e}\n")
        batch_results.append({"prompt": prompt_text, "error": str(e)})